In [12]:
### Rohan's code copied from Colab

# LOAD DATASET
from datasets import load_dataset
import pandas as pd
subtask = "sentiment"
ds = load_dataset("cardiffnlp/tweet_eval", subtask)
labels = ds["train"].features["label"].names

dfs = {split: ds[split].to_pandas() for split in ds}
for split, df in dfs.items():
    df["label_name"] = df["label"].map(lambda i: labels[i])

counts = pd.DataFrame({s: df["label_name"].value_counts() for s, df in dfs.items()})
props = counts / counts.sum()

print(counts, "\n")
print(props.round(3), "\n")

for s in dfs:
  print (s, "majority baseline:", props[s].max())

for df in dfs.values():
    df["n_chars"] = df["text"].str.len()
    df["n_words"] = df["text"].str.split().str.len()

all_df = pd.concat(dfs, names=["split"])
print(all_df[["n_chars", "n_words"]].describe().round(1).drop(index="count"))
print(all_df.groupby("label_name")[["n_chars", "n_words"]].mean().round(1)) # Checking if length differs by class



            train  test  validation
label_name                         
negative     7093  3972         312
neutral     20673  5937         869
positive    17849  2375         819 

            train   test  validation
label_name                          
negative    0.155  0.323       0.156
neutral     0.453  0.483       0.434
positive    0.391  0.193       0.410 

train majority baseline: 0.4532061821769155
test majority baseline: 0.4833116248778899
validation majority baseline: 0.4345
      n_chars  n_words
mean    103.8     18.3
std      27.8      5.4
min       6.0      1.0
25%      85.0     15.0
50%     110.0     19.0
75%     125.0     22.0
max     200.0     35.0
            n_chars  n_words
label_name                  
negative      108.4     19.0
neutral       102.0     17.9
positive      103.5     18.5


In [22]:
# Tokenizer BPE
# !pip install tokenizers
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace

def train_tokenizer(corpus, vocab_size, min_frequency, pretokenizer=Metaspace(), unk_token="[UNK]"):
    """
    corpus: a list of texts (each a string)
    vocab_size: number of distinct BPE tokens
    min_frequency: we only merge token pairs that are frequent enough
    pretokenizer: a simple tokenizer that we use before BPE tokenization

        You probably don't need to change the pretokenizer, unless you are working with
        languages that really don't use any form of blank space
        (then you could, for example, pretokenize at character level)
    """
    # Create an empty BPE tokenizer
    tokenizer = Tokenizer(BPE(unk_token=unk_token)) #unk_token used for characters that cannot be encoded

    # First split the corpus into words
    tokenizer.pre_tokenizer = pretokenizer

    # Tell BPE how large the vocabulary should be
    trainer = BpeTrainer(
        vocab_size=vocab_size, # The size of the vocabulary to learn.
        min_frequency=min_frequency, # Only perform a merge if the pair occurs at least min_frequency times in the training corpus.
        special_tokens=[unk_token]
    )

    # Learn the vocabulary and merge rules
    tokenizer.train_from_iterator(corpus, trainer=trainer)

    return tokenizer



print(ds["train"]["text"])

train_text = ds["train"]["text"]

vocab_size = 5000
min_frequency = 2

bpe_tokenizer = train_tokenizer(train_text, vocab_size=5000, min_frequency=2)

Column(['"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', '"Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"', 'Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.', "Chase Headley's RBI double in the 8th inning off David Price snapped a Yankees streak of 33 consecutive scoreless innings against Blue Jays", '@user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"', ...])





In [28]:
# Representation 1 token indicator
import numpy as np

def token_indicator(text, tokenizer):
    vector = np.zeros(tokenizer.get_vocab_size(), dtype=np.float32)
    token_ids = tokenizer.encode(text).ids
    vector[token_ids] = 1
    return vector

vector = token_indicator("so so good", bpe_tokenizer)
print(vector)
print(vector.shape)                  # (vocab_size,)
print(set(vector.tolist()))          # {0.0, 1.0}
print(vector.sum())                  # number of distinct tokens
print(len(set(bpe_tokenizer.encode("so so good").ids)))  # should match the sum


[0. 0. 0. ... 0. 0. 0.]
(5000,)
{0.0, 1.0}
2.0
2


In [29]:
# Representation 2 bag of words
# Representation 1 token indicator

def bag_of_words(text, tokenizer):
    vector = np.zeros(tokenizer.get_vocab_size(), dtype=np.float32)
    token_ids = tokenizer.encode(text).ids
    for token_id in token_ids:
        vector[token_id] += 1
    return vector

vector = bag_of_words("so so good", bpe_tokenizer)
print(vector)
print(vector.shape)                  # (vocab_size,)
print(set(vector.tolist()))          # {0.0, 1.0}
print(vector.sum())                  # number of distinct tokens


[0. 0. 0. ... 0. 0. 0.]
(5000,)
{0.0, 1.0, 2.0}
3.0


In [30]:
# Representation 3 bi-gram
import numpy as np

def build_bigram_vocab(corpus, tokenizer, max_bigrams=5000):
    counter = {}

    for tweet in corpus:
        ids = tokenizer.encode(tweet).ids
        for i in range(len(ids) - 1):
            pair = (ids[i], ids[i + 1])
            counter[pair] = counter.get(pair, 0) + 1

    # Sort pairs from most to least frequent and keep the top ones
    top_pairs = sorted(counter, key=counter.get, reverse=True)[:max_bigrams]

    # Give each kept pair its own index
    return {pair: index for index, pair in enumerate(top_pairs)}


def bi_gram(text, tokenizer, bigram_vocab):
    vocab_size = tokenizer.get_vocab_size()
    vector = np.zeros(vocab_size + len(bigram_vocab), dtype=np.float32)
    ids = tokenizer.encode(text).ids

    # Token counts go in the first vocab_size slots
    for token_id in ids:
        vector[token_id] += 1

    # Bigram counts go in the slots after that
    for i in range(len(ids) - 1):
        pair = (ids[i], ids[i + 1])
        if pair in bigram_vocab:
            vector[vocab_size + bigram_vocab[pair]] += 1

    return vector

bigram_vocab = build_bigram_vocab(train_text, bpe_tokenizer)

X_train = np.stack([bi_gram(text, bpe_tokenizer, bigram_vocab) for text in train_text])
X_train

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 2., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(45615, 10000), dtype=float32)

In [33]:
import sys, os, copy, json, time, contextlib
from pathlib import Path

pool_texts = list(ds["train"]["text"]) + list(ds["validation"]["text"])
pool_labels = np.array(list(ds["train"]["label"]) + list(ds["validation"]["label"]))
NUM_CLASSES = len(labels)
print(len(pool_texts), "tweets in CV pool")

47615 tweets in CV pool


In [37]:
def encode(encoder_name, texts, bigram_vocab=None):
    """
    Apply encoders to a list of texts -> np.array [n_texts, dim].
    token_indicator() prints the token ids of every tweet, so stdout is silenced here
    (otherwise ~47k lines get printed per fold). The functions themselves are unchanged.
    """
    with open(os.devnull, "w") as sink, contextlib.redirect_stdout(sink):
        if encoder_name == "indicator":
            rows = [token_indicator(t, bpe_tokenizer) for t in texts]
        elif encoder_name == "bow":
            rows = [bag_of_words(t, bpe_tokenizer) for t in texts]
        elif encoder_name == "bigram":
            rows = [bi_gram(t, bpe_tokenizer, bigram_vocab) for t in texts]
        else:
            raise ValueError(encoder_name)
    return np.stack(rows)

_ids = bpe_tokenizer.encode("so so good").ids
print(_ids)

[313, 313, 504]


In [40]:
import scipy.sparse as sp
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score

class SparseDeepMLP(nn.Module):
    """
    Third architecture: the tutorial's SparseMLP with two hidden layers and more dropout.
    """
    def __init__(self, input_dim, hidden_dims=(256, 128), num_classes=3, dropout=0.3):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_dim, hidden_dims[0]))
        self.bias = nn.Parameter(torch.zeros(hidden_dims[0]))
        nn.init.xavier_uniform_(self.weight)
        layers = [nn.ReLU(), nn.Dropout(dropout)]
        for d_in, d_out in zip(hidden_dims[:-1], hidden_dims[1:]):
            layers += [nn.Linear(d_in, d_out), nn.ReLU(), nn.Dropout(dropout)]
        layers += [nn.Linear(hidden_dims[-1], num_classes), nn.LogSoftmax(dim=-1)]
        self.rest = nn.Sequential(*layers)

    def forward(self, x):
        return self.rest(torch.sparse.mm(x, self.weight) + self.bias)

ARCH_CONFIG = {
    "logistic": {"max_iter": 100},                     # same solver as part1/loglin.py
    "mlp":      {"hidden_dim": 128, "dropout": 0.1},   # tutorial SparseMLP
    "deep_mlp": {"hidden_dims": (256, 128), "dropout": 0.3},
}
TRAIN_CONFIG = {"batch_size": 64, "learning_rate": 1e-3, "weight_decay": 1e-4,
                "max_epochs": 10, "patience": 2, "dev_fraction": 0.1}